# Bangladesh: 0147_xee_flood

Static adaptation only; Earth Engine/data/asset execution not verified.

Original scientific assumptions and date windows retained; use the national workflow for QA improvements.

Review scale, missing data, units, and any training/calibration requirements before interpreting results.

May contain tutorial-specific assets or imported training features; inspect dependencies before execution.

In [ ]:
# Bangladesh adaptation: 0147_xee_flood
# Original author: Amirhossein Ahrari; see original attribution below and originals/.
from bdgee.core import initialize, boundaries
from bdgee.notebook import export_outputs, upload_local
initialize()  # Run `earthengine authenticate` first; project is configured in bdgee.
BD_COUNTRY, BD_DISTRICTS = boundaries()
BD_ROI = BD_COUNTRY.geometry()
BD_BBOX = [88.0, 20.5, 92.7, 26.7]  # search only, not analysis boundary
roi = geometry = border = iran = tehran = province = loc = point = BD_ROI
# Run from the repository root; all local outputs are isolated by source.
from pathlib import Path
import os
_REPO_ROOT = Path.cwd()
_OUTPUT_DIR = Path(os.environ.get('BDGEE_OUTPUT_DIR', _REPO_ROOT / 'outputs' / 'flood' / '0147_xee_flood'))
_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(_OUTPUT_DIR)


In [ ]:
"""GEE_0066_flood.ipynb


utorial Code by Amirhossein Ahrari
YouTube: https://www.youtube.com/@amirhosseinahrarigee
Tutorial Video: Google Earth Engine Tutorial-152: Sentinel-1 Flood Detection using Python API (Xee)

This code is part of a tutorial series on Earth Engine programming techniques
presented on the Amirhossein Ahrari YouTube channel. You are free to use and modify
this code for academic and non-academic purposes. Don't forget to subscribe to
the Amirhossein Ahrari channel and follow the videos to support the instructor


"""
import ee
import geemap
import xee
import xarray as xr
import matplotlib.pyplot as plt
None
None
map = geemap.Map(basemap='TERRAIN')
map
loc = BD_ROI
iran = BD_ROI
map.addLayer(iran, {}, 'Iran')
pr = ee.ImageCollection('ECMWF/ERA5/MONTHLY').filterDate('2018', '2020').select('total_precipitation').map(lambda x: x.clip(iran).copyProperties(x, x.propertyNames()))
ds = xr.open_dataset(pr, engine='ee', crs='EPSG:4326', scale=0.27, geometry=BD_ROI)
ds = ds.sortby('time') * 1000.0
plot = ds.total_precipitation.plot.contourf(x='lon', y='lat', col='time', col_wrap=6, robust=True, cmap='Blues')
for ax in plot.axes.flat:
    ax.set_title(ax.get_title(), fontsize=15)
    ax.set_xlabel(ax.get_xlabel(), fontsize=15)
    ax.set_ylabel(ax.get_ylabel(), fontsize=15)
    ax.tick_params(axis='both', labelsize=15)
plot.cbar.set_label('Precipitation [mm]', fontsize=15)
plot.cbar.ax.tick_params(labelsize=15)
plt.savefig('pr_iran.png', bbox_inches='tight', dpi=360)
map2 = geemap.Map(basemap='SATELLITE')
map2
roi = BD_ROI
sen1 = ee.ImageCollection('COPERNICUS/S1_GRD').filterDate('2018', '2020').filterBounds(roi).filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')).filter(ee.Filter.eq('instrumentMode', 'IW')).filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING')).select('VV')
ds2 = xr.open_dataset(sen1, engine='ee', crs='EPSG:4326', scale=0.001, geometry=BD_ROI)
ds2 = ds2.sortby('time') * 1
ds2_monthly = ds2.resample(time='M').min('time')
before = ds2_monthly.sel(time='2019-03').isel(time=0)
after = ds2_monthly.sel(time='2019-04').isel(time=0)
flood = before - after
fig, ax = plt.subplots(1, 3, figsize=(12, 3))
plt.tight_layout(w_pad=3)
plot1 = before.VV.plot(x='lon', y='lat', robust=True, cmap='gray', ax=ax[0])
plot2 = after.VV.plot(x='lon', y='lat', robust=True, cmap='gray', ax=ax[1])
plot3 = flood.VV.plot(x='lon', y='lat', robust=True, cmap='gray', ax=ax[2], vmin=0)
plt.savefig('flood.png', dpi=360, bbox_inches='tight')


In [ ]:

# Review these task plans; task creation does not start exports.
os.chdir(_REPO_ROOT)
bd_tasks = export_outputs(globals(), 'flood', '0147_xee_flood', scale=1000)
# Set an appropriate native scale above before creating plans.
# for task in bd_tasks: task.start()
# upload_local(_OUTPUT_DIR, 'flood')  # explicit upload; requires Drive OAuth scope
